# Round 2: second seed, whole-window spectrum and n_fft 128

The fixes test found that finer STFT bins (`--stft-nfft 64`) were the one change that helped: macro top-1 rose from 35.7% to 38.5%. This round checks that result and tries two ways of seeing finer frequency detail:

1. **Second seed.** `--stft-nfft 64` again with seed 98. If it lands near 38.5%, the gain is real and not one lucky run.
2. **Whole-window spectrum.** `--full-spectrum` adds a branch that reads the 512-point spectrum of the whole window (6.25 kHz bins). It's aimed at OFDM, LTE and NR, SSB against DSB, tones and analog signals. It runs on top of `--stft-nfft 64`, so it compares with that run.
3. **n_fft 128.** 25 kHz bins in the STFT itself. Longer frames can blur FHSS hops, so watch FHSS here.

Same setup as before: all 60 classes, a quarter of every shard, 15 epochs, random placement on. None of these models is an ensemble member. Don't report their scores, and don't save them to `members/`.

Before you start: **Runtime → Change runtime type → GPU**. The last round's runs took about 25 minutes each.

## 1. Get the code

In [ ]:
%cd /content
!rm -rf sedicAI_NEXA
!git clone -b eileen-17class-pilot https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!pip install -q pyyaml h5py gdown

## 2. Check the GPU

This must say `CUDA: True`. If it doesn't, change the runtime type and start again.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Turn on the GPU: Runtime -> Change runtime type'

## 3. Save results to your Drive

New runs go to `OUT`. The compare step also reads the earlier `current` and `stft64` scorecards from the last two tests' folders.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WIDE = '/content/drive/MyDrive/nexa_wide_iq_test'
FIXES = '/content/drive/MyDrive/nexa_fixes_test'
OUT = '/content/drive/MyDrive/nexa_round2_test'
!mkdir -p "$OUT/seed2" "$OUT/spectrum" "$OUT/stft128"
!ls "$FIXES/stft64"

## 4. Download the shards

About 3.8 GB from `processed_data`. Skipped if this session already has them. Training checks every file against its manifest, so a bad download stops the run.

In [ ]:
import os
FOLDERS = {
    'jam': '1hPb-Hi5-RNXHOPB7eqs5o7oSJ66LZ0Fu',
    'pulse': '18jZ_2gJ1HbCloRw4Au6KbPuUrhJyP3MW',
    'spread': '1xr31dYor-rUnXdyXpK512t-HeGmHyugz',
    'radar': '1rGQmzS3dz0-0GE6Ixk7cVkSUtX6daUel',
    'fskpsk': '15aB-sjz_hWMcejUo0JIwd0S2cackg_Y8',
    'analog': '1oz2amRn65zqJ_JWymViaPhZ0wANdFLuZ',
    'multicarrier': '16FsXoJ4yc--pvm3yv89vd8RhTPh3VR2o',
    'apsk': '1uidz9R9NUz91mYe9ko1o8gBzX5Z8sedr',
}
for group, folder in FOLDERS.items():
    if os.path.isdir(f'/content/shards/{group}') and os.listdir(f'/content/shards/{group}'):
        print(f'{group}: already here')
        continue
    !gdown --folder "https://drive.google.com/drive/folders/{folder}" -O "/content/shards/{group}"
!du -sh /content/shards/*

## 5. Check first

A few minutes on a small sample. If this fails, stop and fix it.

In [ ]:
GROUPS = 'jam pulse spread radar fskpsk analog multicarrier apsk'
RUN = f'python scripts/pilot_train_shards.py --shards /content/shards --groups {GROUPS} --fraction 0.25 --epochs 15'
!python scripts/pilot_train_shards.py --shards /content/shards --groups {GROUPS} --out /content/check --fraction 0.005 --epochs 1 --stft-nfft 64 --full-spectrum

## 6. Train

If the session drops, run cells 1 to 4 and the `GROUPS` cell again, then the run that stopped: it resumes from your Drive.

The whole-window spectrum first, because it's the new idea:

In [ ]:
!{RUN} --out "$OUT/spectrum" --stft-nfft 64 --full-spectrum

The second seed, to check the n_fft 64 result:

In [ ]:
!{RUN} --out "$OUT/seed2" --stft-nfft 64 --seed 98

n_fft 128:

In [ ]:
!{RUN} --out "$OUT/stft128" --stft-nfft 128

## 7. Compare

The overall table, then top-1 per family (where the remaining errors are), then every class. The watch list marks FHSS, radar and the classes n_fft 64 hurt. Tell Claude when this is done and it can read the scorecards from your Drive.

In [ ]:
import json
runs = {'current': f'{WIDE}/current', 'stft64': f'{FIXES}/stft64', 'seed2': f'{OUT}/seed2',
        'spectrum': f'{OUT}/spectrum', 'stft128': f'{OUT}/stft128'}
cards = {k: json.load(open(f'{v}/scorecard.json')) for k, v in runs.items() if os.path.exists(f'{v}/scorecard.json')}
print(f"{'run':<10}{'top-1':>8}{'recall 0.5':>12}{'at 0 top-1':>12}")
for k, c in cards.items():
    zero = sum(p['top1_recall'] < 0.02 for p in c['per_class'].values())
    print(f"{k:<10}{c['macro_top1']:>8.3f}{c['macro_recall']:>12.3f}{zero:>12}")

FAMILIES = {'QAM': ['16QAM', '32QAM', '64QAM', '128QAM', '256QAM'],
            'PSK': ['QPSK', 'OQPSK', '8PSK', '16PSK', '32PSK'],
            'APSK': ['16APSK', '32APSK', '64APSK', '128APSK'],
            'AM': ['AM_DSB_WC', 'AM_DSB_SC', 'AM_SSB_WC', 'AM_SSB_SC'],
            'ANGLE': ['WBFM', 'NBFM', 'PM'], 'OFDM': ['OFDM', 'LTE_LIKE', 'NR_LIKE'],
            'TONE': ['JAM_TONE', 'RADAR_CW', 'AFSK']}
print()
print('Top-1 averaged per family')
print(f"{'family':<8}" + ''.join(f'{k:>10}' for k in cards))
for f, members in FAMILIES.items():
    print(f'{f:<8}' + ''.join(f"{sum(cards[k]['per_class'][m]['top1_recall'] for m in members) / len(members):>10.3f}" for k in cards))

WATCH = ['FHSS', 'LFM_RADAR', 'RADAR_FRANK', 'RADAR_FMCW', 'JAM_PARTIAL_BAND', 'OQPSK']
print()
print('Top-1 per class, biggest loss for "spectrum" against stft64 first')
print(f"{'class':<18}" + ''.join(f'{k:>10}' for k in cards))
top = lambda k, n: cards[k]['per_class'][n]['top1_recall']
names = list(cards['current']['per_class'])
order = sorted(names, key=lambda n: top('spectrum', n) - top('stft64', n)) if 'spectrum' in cards else names
for n in order:
    mark = ' <- watch' if n in WATCH else ''
    print(f'{n:<18}' + ''.join(f'{top(k, n):>10.3f}' for k in cards) + mark)